# Synthetic A/B test case study: auto discount vs manual coupon

> **Completed synthetic practice analysis only.** The repository contains no real customer data, no live experiment, and no evidence for a business rollout.

This notebook demonstrates how to validate assignment, compare intent-to-treat (ITT) conversion, and summarize contribution margin and guardrails. The generator deliberately sets purchase probabilities to 18.5% for Control and 20.35% for Treatment. Any apparent Treatment lift is therefore built into the simulation; it is not a causal finding about shoppers.

The run uses the frozen assumptions in `PRE_REGISTRATION.md`. Its outputs are for learning and interview discussion only.


## 1. Load and label data

The committed CSV is synthetic. The analysis below uses all assigned customers as the denominator, including rows without an exposure flag, as specified for the practice ITT analysis.


In [ ]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == 'analysis':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.stats import two_proportion_z_test, difference_ci, proportion_ci

path = Path('data/synthetic_experiment.csv')  # synthetic illustration only
path = path if path.is_absolute() else project_root / path
df = pd.read_csv(path, comment='#')
assert df['data_status'].eq('SYNTHETIC').all(), 'Confirm dataset provenance before analysis'
df.head()

## 2. Validate grain, assignment, and sample ratio

Confirm one row per customer, the expected experiment and variants, and eligibility. Then check whether the assigned group split is unusually far from the planned 50/50 split (sample ratio mismatch, or SRM). In a real experiment, do this before looking at outcomes. Exposure is diagnostic; ITT remains the primary comparison.


In [ ]:
assert df['customer_id'].is_unique, 'Expected one row per randomized customer'
assert set(df['experiment_id'].unique()) == {301}, 'Unexpected experiment ID'
assert set(df['variant'].unique()) == {'control', 'treatment'}, 'Unexpected or missing variant'
assert df['eligible'].all(), 'Unexpected ineligible rows in this practice dataset'
assert df['data_status'].eq('SYNTHETIC').all(), 'Do not analyze without confirming data provenance'

expected_ids = {'control': 3011, 'treatment': 3012}
assert df.groupby('variant')['variant_id'].nunique().eq(1).all()
assert df.groupby('variant')['variant_id'].first().to_dict() == expected_ids
df.groupby('variant').size()


### SRM check

For the 15,000-row synthetic run, 7,500 customers per arm are expected under a 50/50 allocation. The observed counts are 7,561 Control and 7,439 Treatment. The chi-square statistic is about 0.992 (1 degree of freedom), with p ≈ 0.319. This is above the preregistered p < 0.001 investigate threshold, so this run has no SRM flag.

That only checks the generated assignment counts. It does not validate a real assignment system. Exposure is about 96.03% in Control and 96.06% in Treatment; those generated flags are diagnostic only.


In [ ]:
from math import erfc, sqrt

assignment_counts = df['variant'].value_counts().reindex(['control', 'treatment'])
expected_each = len(df) / 2
srm_chi2 = (((assignment_counts - expected_each) ** 2) / expected_each).sum()
srm_p_value = erfc(sqrt(srm_chi2 / 2))  # chi-square(1) survival probability
exposure_rate = df.groupby('variant')['exposed'].mean()

srm_summary = pd.DataFrame({
    'observed': assignment_counts.astype(int),
    'expected_50_50': [expected_each, expected_each],
    'exposure_rate_diagnostic': exposure_rate
})
print(srm_summary)
print(f'chi-square(1)={srm_chi2:.4f}; p={srm_p_value:.4f}; investigate if p < 0.001')
assert srm_p_value >= 0.001, 'Investigate SRM before interpreting outcomes'


The SRM p-value is above 0.001, so the predefined investigate trigger is not met for this synthetic assignment.

## 3. Primary ITT conversion analysis

The denominator is every eligible customer assigned to each group, whether or not the exposure flag is true. The code below calculates counts from the data rather than hardcoding a particular run's counts.


In [ ]:
summary = df.groupby('variant')['converted'].agg(conversions='sum', assigned='count')
summary


In [ ]:
control = summary.loc['control']
treatment = summary.loc['treatment']
x0, n0 = int(control['conversions']), int(control['assigned'])
x1, n1 = int(treatment['conversions']), int(treatment['assigned'])

result = two_proportion_z_test(x0, n0, x1, n1)
ci = difference_ci(x0, n0, x1, n1)
relative_lift = result.difference / result.control_rate
planned_mde_pp = 1.85

print(f'Control: {x0:,}/{n0:,} = {result.control_rate:.2%}')
print(f'Treatment: {x1:,}/{n1:,} = {result.treatment_rate:.2%}')
print(f'Treatment - Control: {result.difference * 100:+.2f} percentage points')
print(f'95% CI: [{ci[0] * 100:+.2f}, {ci[1] * 100:+.2f}] percentage points')
print(f'Relative lift: {relative_lift:.2%}; two-sided p-value: {result.p_value_two_sided:.4f}')
print(f'Planned MDE: +{planned_mde_pp:.2f} percentage points')


### Interpreting the conversion result

In this generated run, Control converted at 18.94% and Treatment at 20.46%, a difference of +1.52 percentage points (95% CI +0.25 to +2.79 pp; two-sided p = 0.0192). The planned +1.85 pp MDE is a sample-size planning assumption, not a promised effect or a pass/fail cutoff. The point estimate is below the MDE, while the interval includes effects both below and above it.

Because the generator was explicitly set to give Treatment a higher purchase probability, this result demonstrates the analysis workflow; it does not show that automatic discounts cause a real lift.


## 4. Contribution margin and guardrails

A higher conversion rate alone does not establish that a discount is worthwhile. Compare average contribution margin across all assigned customers and show uncertainty. Then summarize guardrails descriptively. No real guardrail thresholds are defined for this simulation.


### Contribution margin per assigned customer


In [ ]:
economics = (
    df.groupby('variant')['contribution_margin']
      .agg(mean_per_customer='mean', total='sum', customers='count')
)
economics


In [ ]:
from math import sqrt
from statistics import NormalDist

control_margin = df.loc[df['variant'] == 'control', 'contribution_margin'].astype(float)
treatment_margin = df.loc[df['variant'] == 'treatment', 'contribution_margin'].astype(float)

margin_difference = treatment_margin.mean() - control_margin.mean()
margin_se = sqrt(
    control_margin.var(ddof=1) / len(control_margin)
    + treatment_margin.var(ddof=1) / len(treatment_margin)
)
critical_value = NormalDist().inv_cdf(0.975)  # large-sample 95% interval
margin_ci = (
    margin_difference - critical_value * margin_se,
    margin_difference + critical_value * margin_se
)

print(f'Control mean: ₹{control_margin.mean():.2f} per assigned customer')
print(f'Treatment mean: ₹{treatment_margin.mean():.2f} per assigned customer')
print(f'Treatment - Control: ₹{margin_difference:.2f} per assigned customer')
print(f'Approx. 95% large-sample interval: ₹{margin_ci[0]:.2f} to ₹{margin_ci[1]:.2f}')


The synthetic run gives mean contribution margin of ₹168.15 per assigned customer in Control and ₹169.22 in Treatment: a difference of about **+₹1.08**. The approximate 95% large-sample interval is **−₹10.03 to +₹12.19 per assigned customer**, which includes zero. This is an uncertainty estimate for this synthetic sample, not a real profit estimate.


### Guardrails (descriptive only)


In [ ]:
guardrails = (
    df.groupby('variant')
      .agg(
          customers=('customer_id', 'count'),
          refunds=('refund', 'sum'),
          payment_failures=('payment_failed', 'sum'),
          support_contacts=('support_contact', 'sum')
      )
)

for measure in ['refunds', 'payment_failures', 'support_contacts']:
    guardrails[f'{measure}_rate'] = guardrails[measure] / guardrails['customers']

guardrails


### Case-study readout and limitations

| Measure | Control | Treatment | Treatment − Control |
|---|---:|---:|---:|
| Assigned customers | 7,561 | 7,439 | — |
| Conversion | 18.94% | 20.46% | +1.52 pp (95% CI +0.25 to +2.79; p=0.0192) |
| Contribution margin per assigned customer | ₹168.15 | ₹169.22 | +₹1.08 (approx. 95% interval −₹10.03 to +₹12.19) |
| Refund rate | 0.48% | 0.55% | Descriptive only |
| Payment-failure rate | 1.93% | 2.04% | Descriptive only |
| Support-contact rate | 4.18% | 4.14% | Descriptive only |

**How to read this:** the generator sets the conversion probabilities to 18.5% for Control and 20.35% for Treatment. The realized rates vary around those settings because of randomness. Therefore the positive conversion result is expected by design and is not evidence about real shoppers. The MDE of +1.85 pp was used for planning; it is not a guaranteed minimum result.

**Economics assumptions:** AOV ₹2,500; pre-discount gross margin 40%; discount ₹150; Control coupon redemption 60%; Treatment discount application 100%. The generator assigns zero contribution margin to refunded orders and subtracts no other operating costs. These are illustrative inputs, not Finance-approved values. Guardrail thresholds are not specified, so no acceptability decision is possible.

**Limitations:** all customers, assignments, exposures, outcomes, and business values are synthetic; the seven-day outcome window is not modeled over time; exposure is generated independently from conversion; the customer attributes are not representative. No Ship/Stage/Hold or real rollout recommendation follows from this run.

**For a real experiment:** confirm eligibility and identity rules, use a live randomized assignment and validated event data, set the outcome window and Finance/Product guardrail thresholds in advance, verify the economic estimator, calculate sample size from validated inputs, and complete the preregistered run before deciding.
